In [ ]:
import cellestial as cl
import scanpy as sc
from lets_plot import *

LetsPlot.setup_html()

data = sc.read("data/pbmc3k_pped.h5ad")

In [ ]:
dim = cl.dimensional(
    data,
    dimensions="umap",
    key="leiden",
    size=1,
    axis_type="arrow",
    alpha=0.6,
    tooltips=["leiden", "n_genes", "total_counts_hb", "MT-ND2"],
    legend_ondata=True,
    ondata_size=12,
    ondata_color="black",
    ondata_fontface="bold",
    ondata_family="sans",
    ondata_alpha=0.8,
)
dim += ggsize(800, 600)
dim

In [ ]:
frame = cl.retrieve(dim)

In [ ]:
import polars as pl
from lets_plot import *
from scipy.spatial import ConvexHull
import numpy as np

In [ ]:
def get_hull_frame(frame, x, y, group_by):
    hulls = []
    # Identify unique groups to highlight
    groups = frame.get_column(group_by).unique().to_list()

    for g in groups:
        # Extract coordinates for the specific group
        pts = frame.filter(pl.col(group_by) == g).select([x, y]).to_numpy()

        # We need at least 3 points to create a shape
        if len(pts) > 2:
            hull = ConvexHull(pts)
            # vertices returns the indices of the outer points
            # we repeat the first index at the end to close the loop
            hull_indices = np.append(hull.vertices, hull.vertices[0])
            hull_pts = pts[hull_indices]

            hulls.append(
                pl.DataFrame({x: hull_pts[:, 0], y: hull_pts[:, 1], group_by: [g] * len(hull_pts)})
            )

    return pl.concat(hulls)

In [ ]:
get_hull_frame(frame, "X_UMAP1", "X_UMAP2", "leiden")

In [ ]:
highlighted_hulls = get_hull_frame(
    frame.filter(pl.col("leiden").is_in(["2", "9", "0"])),
    x="X_UMAP1",
    y="X_UMAP2",
    group_by="leiden",
)
highlighted_hulls

In [ ]:
dim + geom_path(
    data=highlighted_hulls,
    mapping=aes(x="X_UMAP1", y="X_UMAP2", group="leiden"),
    linetype="dashed",
)

In [ ]:
dim + geom_density2d(
    data=frame.filter(pl.col("leiden").is_in(["2", "9", "0"])),
    mapping=aes(x="X_UMAP1", y="X_UMAP2"),
    color="black",
    bins=3,
    levels=[0.2, 0.5, 0.8],
    linetype="dashed",
)

In [ ]:
def smooth_hull(frame, x, y, group_by, refinements=3):
    """Calculates a convex hull and applies Chaikin's smoothing."""
    pts = frame.select([x, y]).to_numpy()
    if len(pts) < 3:
        return None

    # 1. Get the initial Hull
    hull = ConvexHull(pts)
    path = pts[np.append(hull.vertices, hull.vertices[0])]

    # 2. Chaikin's Algorithm: Cut the corners
    for _ in range(refinements):
        L = len(path)
        new_path = []
        for i in range(L - 1):
            p0, p1 = path[i], path[i + 1]
            # Create two new points at 1/4 and 3/4 along each edge
            # Q = 0.75*p0 + 0.25*p1
            # R = 0.25*p0 + 0.75*p1
            new_path.append(0.75 * p0 + 0.25 * p1)
            new_path.append(0.25 * p0 + 0.75 * p1)
        # Close the loop
        new_path.append(new_path[0])
        path = np.array(new_path)

    return pl.DataFrame({x: path[:, 0], y: path[:, 1], group_by: [frame[group_by][0]] * len(path)})


# --- Usage ---

# Filter for the group you want and smooth it
target_group = ["2","12","9","7"]
group_frame = frame.filter(pl.col("leiden").is_in([target_group]))
curvy_hull = smooth_hull(group_frame, "X_UMAP1", "X_UMAP2", "leiden", refinements=4)


In [ ]:
dim + geom_path(
    data=curvy_hull,
    #mapping=aes(x="X_UMAP1", y="X_UMAP2", group="leiden"),
    linetype="dashed",
    size=0.6,
)

In [ ]:
dim + geom_path(
    data=highlighted_hulls,
    mapping=aes(x="X_UMAP1", y="X_UMAP2", group="leiden"),
    linetype="dashed",
)

In [ ]:
import polars as pl
import numpy as np
from scipy.spatial import ConvexHull
from lets_plot import *

def get_smooth_hull_frame(frame, x, y, group_by, padding=0.5, refinements=4):
    hulls = []
    groups = frame.get_column(group_by).unique().to_list()

    for g in groups:
        pts = frame.filter(pl.col(group_by) == g).select([x, y]).to_numpy()
        if len(pts) < 3: continue

        # 1. Generate the Hull
        hull = ConvexHull(pts)
        
        # 2. Use 'equations' to get unit normals (Ax + By + C = 0)
        # The first two columns of 'equations' are the (A, B) normal vectors
        normals = hull.equations[:, :2]
        
        # 3. Expand vertices using the normals to create 'padding'
        # We find the vertices and offset them so the line doesn't touch the points
        expanded_pts = []
        for i, vertex_idx in enumerate(hull.vertices):
            # Each vertex is shared by two facets; we average their normals
            n1 = normals[i]
            n2 = normals[i - 1]
            avg_normal = (n1 + n2) / np.linalg.norm(n1 + n2)
            expanded_pts.append(pts[vertex_idx] + avg_normal * padding)
        
        # Close the loop
        expanded_pts.append(expanded_pts[0])
        path = np.array(expanded_pts)

        # 4. Increase "dimensions" (interpolation) for smoothing
        # This is Chaikin's algorithm to turn sharp corners into curves
        for _ in range(refinements):
            new_path = []
            for j in range(len(path) - 1):
                p0, p1 = path[j], path[j+1]
                new_path.append(0.75 * p0 + 0.25 * p1)
                new_path.append(0.25 * p0 + 0.75 * p1)
            new_path.append(new_path[0])
            path = np.array(new_path)

        hulls.append(
            pl.DataFrame({
                x: path[:, 0], 
                y: path[:, 1], 
                group_by: [g] * len(path)
            })
        )

    return pl.concat(hulls)


In [ ]:
hull_frame = get_smooth_hull_frame(frame, "X_UMAP1", "X_UMAP2", "leiden", padding=0.8, refinements=15)

In [ ]:
hull_frame

In [ ]:
dim + geom_path(
    data=curvy_hull,
    mapping=aes(x="X_UMAP1", y="X_UMAP2", group="leiden"),
    linetype="dashed",
    size=0.6,
)